In [1]:
import pandas as pd

# Cargar Datos

In [2]:
df = pd.read_csv(
    "../data/results.csv",
    parse_dates=["date"]
)

df.head(10)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False
5,1876-03-25,Scotland,Wales,4.0,0.0,Friendly,Glasgow,Scotland,False
6,1877-03-03,England,Scotland,1.0,3.0,Friendly,London,England,False
7,1877-03-05,Wales,Scotland,0.0,2.0,Friendly,Wrexham,Wales,False
8,1878-03-02,Scotland,England,7.0,2.0,Friendly,Glasgow,Scotland,False
9,1878-03-23,Scotland,Wales,9.0,0.0,Friendly,Glasgow,Scotland,False


# Limpieza de datos

In [3]:
df = df[
    df["home_team"] != df["away_team"]
]

df = df.sort_values(
    "date"
)

df.head(10)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False
5,1876-03-25,Scotland,Wales,4.0,0.0,Friendly,Glasgow,Scotland,False
6,1877-03-03,England,Scotland,1.0,3.0,Friendly,London,England,False
7,1877-03-05,Wales,Scotland,0.0,2.0,Friendly,Wrexham,Wales,False
8,1878-03-02,Scotland,England,7.0,2.0,Friendly,Glasgow,Scotland,False
9,1878-03-23,Scotland,Wales,9.0,0.0,Friendly,Glasgow,Scotland,False


## Separar datos históricos y partidos futuros

In [4]:
historical_df = df.dropna(
    subset=[
        "home_score",
        "away_score"
    ]
).copy()

future_matches = df[
    df["home_score"].isna()
].copy()

historical_df["home_score"] = (
    historical_df["home_score"]
    .astype(int)
)

historical_df["away_score"] = (
    historical_df["away_score"]
    .astype(int)
)

historical_df = historical_df[
    historical_df["date"] >= "2010-01-01"
]

historical_df.head(10)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
33589,2010-01-02,Syria,Zimbabwe,6,0,Friendly,Kuala Lumpur,Malaysia,True
33590,2010-01-02,Yemen,Tajikistan,0,1,Friendly,Sana'a,Yemen,False
33587,2010-01-02,Iran,North Korea,1,0,Friendly,Doha,Qatar,True
33588,2010-01-02,Qatar,Mali,0,0,Friendly,Doha,Qatar,False
33591,2010-01-03,Angola,Gambia,1,1,Friendly,Vila Real de Santo António,Portugal,True
33592,2010-01-04,Egypt,Mali,1,0,Friendly,Dubai,United Arab Emirates,True
33593,2010-01-04,Tanzania,Ivory Coast,0,1,Friendly,Dar es Salaam,Tanzania,False
33594,2010-01-05,Ghana,Malawi,0,0,Friendly,Manzini,Swaziland,True
33605,2010-01-06,Yemen,Japan,2,3,AFC Asian Cup qualification,Sana'a,Yemen,False
33604,2010-01-06,United Arab Emirates,Malaysia,1,0,AFC Asian Cup qualification,Dubai,United Arab Emirates,False


# Información

In [5]:
print("Primer partido:")
print(
    historical_df["date"].min()
)

print("\nÚltimo partido:")
print(
    historical_df["date"].max()
)

print("\nPartidos históricos:")
print(
    historical_df.shape
)

print("\nPartidos futuros:")
print(
    future_matches.shape
)

print(
    historical_df["tournament"]
    .value_counts()
    .head(30)
)

Primer partido:
2010-01-02 00:00:00

Último partido:
2026-05-31 00:00:00

Partidos históricos:
(15694, 9)

Partidos futuros:
(72, 9)
tournament
Friendly                                 4965
FIFA World Cup qualification             3422
African Cup of Nations qualification     1052
UEFA Euro qualification                  1016
UEFA Nations League                       658
CONCACAF Nations League                   422
African Cup of Nations                    365
AFC Asian Cup qualification               269
FIFA World Cup                            256
Gold Cup                                  225
COSAFA Cup                                201
UEFA Euro                                 184
Copa América                              170
AFC Asian Cup                             166
AFF Championship                          150
Island Games                              149
CFU Caribbean Cup qualification           144
CECAFA Cup                                142
Gulf Cup                    

# ELO

In [6]:
INITIAL_ELO = 1500

TOURNAMENT_W = {

    "Friendly": 0.33,

    "FIFA World Cup qualification": 0.45,
    "UEFA Euro qualification": 0.5,

    "African Cup of Nations qualification": 0.45,
    "AFC Asian Cup qualification": 0.45,

    "UEFA Nations League": 0.8,
    "CONCACAF Nations League": 0.6,

    "Gold Cup": 0.6,
    "AFC Asian Cup": 0.65,
    "African Cup of Nations": 0.7,

    "Copa América": 0.8,
    "UEFA Euro": 0.85,

    "FIFA World Cup": 1
}

In [7]:
def get_w_factor(tournament):
    # Los torneos no listados caen en "torneos menores" (0.5) según el paper
    return TOURNAMENT_W.get(tournament, 0.5)

In [8]:
def margin_multiplier(home_goals, away_goals, gamma):
    delta = abs(home_goals - away_goals)
    # El paper utiliza (1 + delta)^gamma
    return (1 + delta) ** gamma

In [9]:
def expected_score(rating_a, rating_b):
    # El paper añade la ventaja de casa en la regresión Logit final, NO en el Elo
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))

In [10]:
def actual_score(home_goals, away_goals):
    """
    Resultado real:
    victoria = 1
    empate = 0.5
    derrota = 0
    """

    if home_goals > away_goals:
        return 1.0, 0.0

    if home_goals < away_goals:
        return 0.0, 1.0

    return 0.5, 0.5

In [11]:
def update_ratings(home_rating, away_rating, home_goals, away_goals, tournament, k0, gamma):
    exp_home = expected_score(home_rating, away_rating)
    exp_away = 1 - exp_home

    act_home, act_away = actual_score(home_goals, away_goals)

    w_i = get_w_factor(tournament)
    margin = margin_multiplier(home_goals, away_goals, gamma)
    
    # La fórmula clave del paper: k = k0 * w_i * (1 + delta)^gamma
    k = k0 * w_i * margin

    new_home = home_rating + k * (act_home - exp_home)
    new_away = away_rating + k * (act_away - exp_away)

    return new_home, new_away

In [12]:
def calculate_elo_history(matches, k0=40, gamma=0.5):
    """
    Se eliminó la degradación de años (years_old) porque el paper 
    solo usa la naturaleza secuencial del Elo para la recencia.
    """
    ratings = {}
    records = []
    
    matches = matches.sort_values("date")

    for _, row in matches.iterrows():
        home = row["home_team"]
        away = row["away_team"]

        home_rating = ratings.get(home, INITIAL_ELO)
        away_rating = ratings.get(away, INITIAL_ELO)

        records.append({
            "date": row["date"],
            "home_team": home,
            "away_team": away,
            "home_elo": home_rating,
            "away_elo": away_rating,
            "tournament": row["tournament"],
            "neutral": row["neutral"]
        })

        new_home, new_away = update_ratings(
            home_rating,
            away_rating,
            row["home_score"],
            row["away_score"],
            row["tournament"],
            k0,      # Pasamos el parámetro k0
            gamma    # Pasamos el parámetro gamma
        )

        ratings[home] = new_home
        ratings[away] = new_away

    elo_df = pd.DataFrame(records)
    return elo_df, ratings

In [13]:
elo_history, current_ratings = (
    calculate_elo_history(
        historical_df
    )
)

## Resultados ELO

In [14]:
print("\nPrimeras filas Elo")

print(elo_history.head())

print("\nÚltimas filas Elo")

print(elo_history.tail())


Primeras filas Elo
        date home_team    away_team  home_elo  away_elo tournament  neutral
0 2010-01-02     Syria     Zimbabwe    1500.0    1500.0   Friendly     True
1 2010-01-02     Yemen   Tajikistan    1500.0    1500.0   Friendly    False
2 2010-01-02      Iran  North Korea    1500.0    1500.0   Friendly     True
3 2010-01-02     Qatar         Mali    1500.0    1500.0   Friendly    False
4 2010-01-03    Angola       Gambia    1500.0    1500.0   Friendly     True

Últimas filas Elo
            date      home_team away_team     home_elo     away_elo  \
15689 2026-05-31        Germany   Finland  1883.925616  1476.384056   
15690 2026-05-31         Poland   Ukraine  1648.941903  1666.135608   
15691 2026-05-31      Singapore  Mongolia  1406.924900  1226.273085   
15692 2026-05-31    Switzerland    Jordan  1790.696984  1732.235183   
15693 2026-05-31  United States   Senegal  1796.929004  1855.343134   

      tournament  neutral  
15689   Friendly    False  
15690   Friendly    Fa

### Top 20 Actual

In [15]:
ranking = (pd.DataFrame(current_ratings.items(),columns=["team","elo"]).sort_values("elo",ascending=False))

print("\nTOP 20 ELO")

print(ranking.head(20))


TOP 20 ELO
            team          elo
93         Spain  2070.901888
51     Argentina  2023.920646
92        France  1983.570366
160      Morocco  1933.310365
91       England  1921.170039
69        Brazil  1915.174029
100     Portugal  1913.268842
31         Japan  1912.728367
142     Colombia  1894.214719
106  Netherlands  1888.172243
85       Germany  1886.504863
62        Mexico  1849.884393
87       Senegal  1844.454658
72       Croatia  1839.185591
4           Iran  1831.017416
132      Ecuador  1826.657257
115      Uruguay  1826.244289
23     Australia  1823.554166
90         Italy  1818.676139
39   South Korea  1818.058959


# Algoritmo iterativo ODM

In [16]:
import numpy as np
def scale(A, x):
    # Usamos np.maximum para evitar divisiones por cero absolutas
    x_inv = 1.0 / np.maximum(x, 1e-6)
    
    # y <- A * (1/x)
    y = A @ x_inv
    
    y_inv = 1.0 / np.maximum(y, 1e-6)
    
    # x_post <- A^T * (1/y)
    x_post = A.T @ y_inv
    
    # y_post <- A * (1/x_post) [Corregido del paper para convergencia real]
    y_post = A @ (1.0 / np.maximum(x_post, 1e-6))
    
    return x_post, y_post

In [17]:
def compute_post_game_odm(o_pre, d_pre, home_goals, away_goals, nb_iter=10, epsilon=0.01):
    """
    o_pre: [ataque_local_pre, ataque_visitante_pre]
    d_pre: [defensa_local_pre, defensa_visitante_pre]
    """
    # Matriz A_ij: goles marcados por el equipo j contra el equipo i
    # Índice 0: Local, Índice 1: Visitante
    # A[0, 1] = goles del visitante contra el local
    # A[1, 0] = goles del local contra el visitante
    A = np.array([
        [epsilon, away_goals + epsilon],
        [home_goals + epsilon, epsilon]
    ])
    
    o_post = np.array(o_pre, dtype=float)
    d_post = np.array(d_pre, dtype=float)
    
    for _ in range(nb_iter):
        # Escalar ataque
        o_rel_o, d_rel_o = scale(A, o_post)
        
        # Escalar defensa (Matriz transpuesta)
        d_rel_d, o_rel_d = scale(A.T, d_post)
        
        # Actualizar
        o_post = o_rel_o + o_rel_d
        d_post = d_rel_o + d_rel_d
        
    return o_post[0], o_post[1], d_post[0], d_post[1]

In [18]:
INITIAL_ODM = 1.0

def calculate_combined_history(matches, k0=40, gamma=0.5):
    ratings_elo = {}
    odm_o_weighted_sum = {}
    odm_d_weighted_sum = {}
    odm_weights_sum = {}
    
    records = []
    
    # Ordenamos y mantenemos el índice
    matches = matches.sort_values("date")

    # OPTIMIZACIÓN VITAL: Usamos itertuples() en lugar de iterrows()
    for row in matches.itertuples():
        # En itertuples, accedemos a las columnas como atributos
        home = row.home_team
        away = row.away_team
        tournament = row.tournament
        home_score = row.home_score
        away_score = row.away_score
        neutral = row.neutral
        
        # 1. RECUPERAR ELO PRE-PARTIDO
        home_elo = ratings_elo.get(home, INITIAL_ELO)
        away_elo = ratings_elo.get(away, INITIAL_ELO)
        
        # 2. RECUPERAR ODM PRE-PARTIDO
        if home in odm_weights_sum and odm_weights_sum[home] > 0:
            home_o_pre = odm_o_weighted_sum[home] / odm_weights_sum[home]
            home_d_pre = odm_d_weighted_sum[home] / odm_weights_sum[home]
        else:
            home_o_pre, home_d_pre = INITIAL_ODM, INITIAL_ODM
            
        if away in odm_weights_sum and odm_weights_sum[away] > 0:
            away_o_pre = odm_o_weighted_sum[away] / odm_weights_sum[away]
            away_d_pre = odm_d_weighted_sum[away] / odm_weights_sum[away]
        else:
            away_o_pre, away_d_pre = INITIAL_ODM, INITIAL_ODM

        # Guardar en el registro (Añadidos los goles reales)
        records.append({
            "date": row.date,
            "home_team": home,
            "away_team": away,
            "home_score": home_score,
            "away_score": away_score,
            "home_elo": home_elo,
            "away_elo": away_elo,
            "home_o": home_o_pre,
            "home_d": home_d_pre,
            "away_o": away_o_pre,
            "away_d": away_d_pre,
            "tournament": tournament,
            "neutral": neutral
        })

        # 3. ACTUALIZAR ELO POST-PARTIDO
        new_home_elo, new_away_elo = update_ratings(
            home_elo, away_elo,
            home_score, away_score,
            tournament, k0, gamma
        )
        ratings_elo[home] = new_home_elo
        ratings_elo[away] = new_away_elo
        
        # 4. ACTUALIZAR ODM POST-PARTIDO
        h_o_post, a_o_post, h_d_post, a_d_post = compute_post_game_odm(
            [home_o_pre, away_o_pre],
            [home_d_pre, away_d_pre],
            home_score, away_score
        )
        
        # 5. ACTUALIZAR PROMEDIOS PONDERADOS
        w_i = get_w_factor(tournament)
        
        for team in [home, away]:
            if team not in odm_weights_sum:
                odm_o_weighted_sum[team] = 0.0
                odm_d_weighted_sum[team] = 0.0
                odm_weights_sum[team] = 0.0
                
        odm_o_weighted_sum[home] += h_o_post * w_i
        odm_d_weighted_sum[home] += h_d_post * w_i
        odm_weights_sum[home] += w_i
        
        odm_o_weighted_sum[away] += a_o_post * w_i
        odm_d_weighted_sum[away] += a_d_post * w_i
        odm_weights_sum[away] += w_i

    # Devolvemos el DataFrame respetando el índice original de "matches"
    return pd.DataFrame(records, index=matches.index)

# Función RPS

In [19]:
import numpy as np

def calculate_rps(probs, actual_results):
    """
    probs: Array de shape (N, 3) con probabilidades [p_derrota, p_empate, p_victoria]
    actual_results: Array 1D de shape (N,) con los resultados reales (0=Derrota, 1=Empate, 2=Victoria)
    """
    N = len(actual_results)
    
    # Convertir resultados reales a One-Hot Encoding [y_derrota, y_empate, y_victoria]
    y = np.zeros((N, 3))
    y[np.arange(N), actual_results] = 1.0
    
    # Calcular las sumas acumuladas (CDF) para las predicciones y los valores reales
    # k=1: Probabilidad de derrota (vs Realidad de derrota)
    # k=2: Probabilidad de derrota o empate (vs Realidad de derrota o empate)
    cum_probs = np.cumsum(probs, axis=1)
    cum_y = np.cumsum(y, axis=1)
    
    # La sumatoria de las diferencias al cuadrado, dividida por |r|-1 (que es 2)
    rps = np.sum((cum_probs[:, :2] - cum_y[:, :2]) ** 2, axis=1) / 2.0
    
    return np.mean(rps)

# Modelo Predictivo: Regresión Logit Ordenada

In [20]:
def ordered_logit_probabilities(X_dot_beta, theta_0, d_theta):
    """
    X_dot_beta: El resultado de multiplicar las características por sus coeficientes.
    theta_0: El primer punto de corte (separa Derrota de Empate).
    d_theta: Distancia exponencial al segundo punto de corte (asegura que theta_1 > theta_0).
    """
    theta_1 = theta_0 + np.exp(d_theta) # Restricción matemática obligatoria
    
    # Probabilidad acumulada de Derrota
    p_loss = 1.0 / (1.0 + np.exp(-(theta_0 - X_dot_beta)))
    
    # Probabilidad acumulada de Derrota o Empate
    p_loss_tie = 1.0 / (1.0 + np.exp(-(theta_1 - X_dot_beta)))
    
    # Probabilidades individuales
    p_tie = p_loss_tie - p_loss
    p_win = 1.0 - p_loss_tie
    
    # Nos aseguramos de no devolver probabilidades negativas por errores de precisión
    probs = np.column_stack((p_loss, p_tie, p_win))
    return np.clip(probs, 1e-15, 1.0)

# Optimizador L-BFGS-B (Optimización Conjunta)

In [21]:
from scipy.optimize import minimize

def encode_result_ordinal(home_score, away_score):
    if home_score < away_score: return 0 # Derrota local
    if home_score == away_score: return 1 # Empate
    return 2 # Victoria local

def objective_function(params, historical_matches, validation_indices):
    # 1. Desempaquetar los parámetros a optimizar
    k0, gamma, beta_elo, beta_odm1, beta_odm2, beta_home, theta_0, d_theta = params
    
    # 2. Recalcular toda la historia con los nuevos k0 y gamma
    # (Asume que tienes la función `calculate_combined_history` del paso anterior)
    df_history = calculate_combined_history(historical_matches, k0=k0, gamma=gamma)
    
    # 3. Extraer solo el set de validación (por ejemplo, los partidos de un Mundial específico)
    val_df = df_history.loc[validation_indices].copy()
    
    # 4. Construir el vector X * Beta
    elo_diff = (val_df['home_elo'] - val_df['away_elo']) / 100.0  # Escalado para estabilidad
    odm_o_d = val_df['home_o'] - val_df['away_d']
    odm_d_o = val_df['home_d'] - val_df['away_o']
    home_adv = (~val_df['neutral']).astype(float)
    
    X_dot_beta = (beta_elo * elo_diff + 
                  beta_odm1 * odm_o_d + 
                  beta_odm2 * odm_d_o + 
                  beta_home * home_adv)
    
    # 5. Predecir probabilidades
    probs = ordered_logit_probabilities(X_dot_beta.values, theta_0, d_theta)
    
    # 6. Obtener resultados reales
    y_true = np.array([encode_result_ordinal(row['home_score'], row['away_score']) 
                       for _, row in val_df.iterrows()])
    
    # 7. Calcular y retornar el RPS
    rps = calculate_rps(probs, y_true)
    
    return rps

def train_joint_model(historical_matches, validation_indices):
    """
    Ejecuta el algoritmo L-BFGS-B para encontrar los parámetros óptimos.
    """
    # Valores iniciales razonables (Initial guess)
    # [k0, gamma, beta_elo, beta_odm1, beta_odm2, beta_home, theta_0, d_theta]
    initial_params = [40.0, 0.5, 0.5, 0.5, 0.5, 0.3, -0.5, 0.0]
    
    # Límites para el optimizador (Bounds)
    bounds = [
        (10.0, 100.0), # k0: factor base de recencia
        (0.0, 2.0),    # gamma: impacto del margen de victoria
        (None, None),  # beta_elo
        (None, None),  # beta_odm1
        (None, None),  # beta_odm2
        (0.0, None),   # beta_home: La ventaja de localía debe ser positiva
        (None, None),  # theta_0
        (-5.0, 5.0)    # d_theta: restringe la distancia entre cortes
    ]
    
    print("Iniciando optimización conjunta (L-BFGS-B)... Esto puede tardar.")
    
    result = minimize(
        objective_function,
        x0=initial_params,
        args=(historical_matches, validation_indices),
        method='L-BFGS-B',
        bounds=bounds,
        options={'disp': True, 'maxiter': 50}
    )
    
    print(f"Optimización completada. RPS Final: {result.fun:.4f}")
    
    return {
        'k0': result.x[0],
        'gamma': result.x[1],
        'beta_elo': result.x[2],
        'beta_odm1': result.x[3],
        'beta_odm2': result.x[4],
        'beta_home': result.x[5],
        'theta_0': result.x[6],
        'd_theta': result.x[7]
    }

In [22]:
# Suponiendo que historical_df tiene los partidos ordenados y con índice reseteado
val_mask = (historical_df["date"] >= "2014-06-01") & (historical_df["date"] <= "2014-07-15")
validation_indices = historical_df[val_mask].index

# Entrenar el modelo
optimal_parameters = train_joint_model(historical_df, validation_indices)

print("Parámetros óptimos encontrados por el modelo:")
print(optimal_parameters)

Iniciando optimización conjunta (L-BFGS-B)... Esto puede tardar.


C:\Users\daniel.cejas\AppData\Local\Temp\ipykernel_9688\2624002254.py:64: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  result = minimize(


Optimización completada. RPS Final: 0.1628
Parámetros óptimos encontrados por el modelo:
{'k0': np.float64(40.01898822546239), 'gamma': np.float64(1.7142655212755424), 'beta_elo': np.float64(0.4561425652538327), 'beta_odm1': np.float64(-0.01657696098170181), 'beta_odm2': np.float64(-0.011444795263682117), 'beta_home': np.float64(0.727719748848296), 'theta_0': np.float64(-0.4380741862587575), 'd_theta': np.float64(0.09775506290503407)}


In [23]:
optimal_params = {
    'k0': 40.01898822546239, 
    'gamma': 1.7142655212755424, 
    'beta_elo': 0.4561425652538327, 
    'beta_odm1': -0.01657696098170181, 
    'beta_odm2': -0.011444795263682117, 
    'beta_home': 0.727719748848296, 
    'theta_0': -0.4380741862587575, 
    'd_theta': 0.09775506290503407
}

# Recalcular el historial final con los parámetros óptimos

In [24]:
print("Calculando ratings finales...")
df_history_final = calculate_combined_history(
    historical_df, 
    k0=optimal_params['k0'], 
    gamma=optimal_params['gamma']
)

# 3. EXTRAER EL "ROSTER" FINAL (El último rating conocido de cada país)
latest_ratings = {}

# Obtenemos todos los equipos únicos
all_teams = pd.concat([df_history_final['home_team'], df_history_final['away_team']]).unique()

for team in all_teams:
    # Filtramos los partidos donde jugó este equipo
    team_matches = df_history_final[(df_history_final['home_team'] == team) | (df_history_final['away_team'] == team)]
    
    if len(team_matches) > 0:
        # Tomamos el último partido
        last_match = team_matches.iloc[-1]
        
        if last_match['home_team'] == team:
            latest_ratings[team] = {
                'elo': last_match['home_elo'], 
                'o': last_match['home_o'], 
                'd': last_match['home_d']
            }
        else:
            latest_ratings[team] = {
                'elo': last_match['away_elo'], 
                'o': last_match['away_o'], 
                'd': last_match['away_d']
            }
            
print(f"Ratings calculados para {len(latest_ratings)} selecciones.")

Calculando ratings finales...
Ratings calculados para 312 selecciones.


# Predictor final

In [26]:
def predict_match_final(team_home, team_away, ratings_dict, params, neutral=True):
    # 1. Extraer ratings actuales
    h_elo, h_o, h_d = ratings_dict[team_home]['elo'], ratings_dict[team_home]['o'], ratings_dict[team_home]['d']
    a_elo, a_o, a_d = ratings_dict[team_away]['elo'], ratings_dict[team_away]['o'], ratings_dict[team_away]['d']
    
    # 2. Calcular diferencias (covariables)
    elo_diff = (h_elo - a_elo) / 100.0
    odm_o_d = h_o - a_d  # Ataque local vs Defensa visitante
    odm_d_o = h_d - a_o  # Defensa local vs Ataque visitante
    home_adv = 0.0 if neutral else 1.0
    
    # 3. Aplicar los coeficientes (Betas)
    X_dot_beta = (params['beta_elo'] * elo_diff + 
                  params['beta_odm1'] * odm_o_d + 
                  params['beta_odm2'] * odm_d_o + 
                  params['beta_home'] * home_adv)
    
    # 4. Obtener probabilidades con la fórmula ordinal
    # (Usamos la función ordered_logit_probabilities que programamos antes)
    probs = ordered_logit_probabilities(np.array([X_dot_beta]), params['theta_0'], params['d_theta'])[0]
    
    return {
        "away_win": probs[0], # Índice 0 = Derrota Local
        "draw": probs[1],     # Índice 1 = Empate
        "home_win": probs[2]  # Índice 2 = Victoria Local
    }

# Simulación MonteCarlo un partido

In [28]:
def simulate_match(team_home, team_away, ratings, params, neutral=True):
    probs = predict_match_final(team_home, team_away, ratings, params, neutral)
    
    outcome = np.random.choice(
        ["away_win", "draw", "home_win"],
        p=[probs["away_win"], probs["draw"], probs["home_win"]]
    )
    return outcome

In [29]:
# Asegúrate de tener cargados 'latest_ratings' y 'optimal_params' del paso anterior

equipo_local = "Spain"
equipo_visitante = "France"

# Obtenemos un resultado simulado (puede ser "home_win", "draw" o "away_win")
resultado_simulado = simulate_match(
    equipo_local,
    equipo_visitante,
    latest_ratings,  # Pasamos el diccionario completo de ratings
    optimal_params,  # Pasamos los parámetros optimizados
    neutral=True
)

print(f"Partido: {equipo_local} vs {equipo_visitante}")
print(f"Resultado de la simulación: {resultado_simulado}")

# --- Opcional: Ver las probabilidades detrás de la simulación ---
probabilidades = predict_match_final(
    equipo_local, 
    equipo_visitante, 
    latest_ratings, 
    optimal_params, 
    neutral=True
)

print("\nProbabilidades matemáticas del modelo:")
print(f"Victoria {equipo_local} (home_win): {probabilidades['home_win'] * 100:.2f}%")
print(f"Empate (draw): {probabilidades['draw'] * 100:.2f}%")
print(f"Victoria {equipo_visitante} (away_win): {probabilidades['away_win'] * 100:.2f}%")

Partido: Spain vs France
Resultado de la simulación: draw

Probabilidades matemáticas del modelo:
Victoria Spain (home_win): 32.09%
Empate (draw): 26.65%
Victoria France (away_win): 41.27%


In [30]:
tests = [
    ("Spain", "France"),
    ("Argentina", "Brazil"),
    ("San Marino", "Argentina"),
    ("Andorra", "Spain"),
    ("Brazil", "Japan"),
    ("England", "Germany")
]

for home, away in tests:
    
    # 1. Llamamos a la nueva función predictora
    # Fíjate que le pasamos los NOMBRES de los equipos, no el rating individual
    pred = predict_match_final(
        team_home=home,
        team_away=away,
        ratings_dict=latest_ratings,
        params=optimal_params,
        neutral=True
    )

    # 2. Imprimimos los resultados con un formato limpio (porcentajes)
    print()
    print(f"--- {home} vs {away} ---")
    print(f"Victoria {home:<12}: {pred['home_win'] * 100:>5.1f}%")
    print(f"Empate              : {pred['draw'] * 100:>5.1f}%")
    print(f"Victoria {away:<12}: {pred['away_win'] * 100:>5.1f}%")


--- Spain vs France ---
Victoria Spain       :  32.1%
Empate              :  26.6%
Victoria France      :  41.3%

--- Argentina vs Brazil ---
Victoria Argentina   :  46.3%
Empate              :  25.9%
Victoria Brazil      :  27.8%

--- San Marino vs Argentina ---
Victoria San Marino  :   0.0%
Empate              :   0.0%
Victoria Argentina   : 100.0%

--- Andorra vs Spain ---
Victoria Andorra     :   0.0%
Empate              :   0.0%
Victoria Spain       : 100.0%

--- Brazil vs Japan ---
Victoria Brazil      :  21.8%
Empate              :  23.8%
Victoria Japan       :  54.3%

--- England vs Germany ---
Victoria England     :  56.0%
Empate              :  23.3%
Victoria Germany     :  20.7%


In [31]:
from collections import Counter
counter = Counter()

N = 10000

# Definimos los equipos que queremos probar
equipo_local = "Spain"
equipo_visitante = "France"

print(f"Ejecutando {N} simulaciones para {equipo_local} vs {equipo_visitante}...")

for _ in range(N):
    # 1. Llamamos a simulate_match con los NUEVOS argumentos
    result = simulate_match(
        equipo_local,        # Pasamos el string del nombre
        equipo_visitante,    # Pasamos el string del nombre
        latest_ratings,      # Tu diccionario con la "foto final"
        optimal_params,      # Tus parámetros optimizados
        neutral=True
    )

    counter[result] += 1

# Imprimimos los resultados agregados
print("\nResultados de la Simulación Monte Carlo:")
for k, v in counter.items():
    print(f"{k}: {round(v / N * 100, 2)}%")

Ejecutando 10000 simulaciones para Spain vs France...

Resultados de la Simulación Monte Carlo:
home_win: 32.1%
draw: 26.6%
away_win: 41.3%


# Simulación MonteCarlo un grupo

In [32]:
import numpy as np

# 1. Tu generador de goles (Se mantiene intacto)
def sample_score(result):
    if result == "draw":
        goals = np.random.choice(
            [0, 1, 2],
            p=[0.4, 0.4, 0.2]
        )
        return goals, goals

    if result == "home_win":
        home = np.random.choice(
            [1, 2, 3, 4],
            p=[0.35, 0.35, 0.2, 0.1]
        )
        away = np.random.randint(0, home)
        return home, away

    away = np.random.choice(
        [1, 2, 3, 4],
        p=[0.35, 0.35, 0.2, 0.1]
    )
    home = np.random.randint(0, away)
    return home, away

# 2. Tu creador de tabla (Se mantiene intacto)
def create_group_table(teams):
    return {
        team: {
            "points": 0,
            "gf": 0,
            "ga": 0,
            "gd": 0
        }
        for team in teams
    }

# 3. Tu actualizador de tabla (Se mantiene intacto)
def update_table(table, home, away, home_goals, away_goals):
    table[home]["gf"] += home_goals
    table[home]["ga"] += away_goals

    table[away]["gf"] += away_goals
    table[away]["ga"] += home_goals

    table[home]["gd"] = table[home]["gf"] - table[home]["ga"]
    table[away]["gd"] = table[away]["gf"] - table[away]["ga"]

    if home_goals > away_goals:
        table[home]["points"] += 3
    elif away_goals > home_goals:
        table[away]["points"] += 3
    else:
        table[home]["points"] += 1
        table[away]["points"] += 1

# 4. Simulador de grupo actualizado con la nueva matemática
def simulate_group_hybrid(teams, ratings, params):
    table = create_group_table(teams)

    for i in range(len(teams)):
        for j in range(i + 1, len(teams)):
            home = teams[i]
            away = teams[j]

            # A. El Logit Ordenado decide QUIÉN gana
            result = simulate_match(
                home,
                away,
                ratings,
                params,
                neutral=True
            )

            # B. Tu función inventa los GOLES
            home_goals, away_goals = sample_score(result)

            # C. Actualizamos
            update_table(table, home, away, home_goals, away_goals)

    # Ordenamos con reglas FIFA reales + Elo como desempate final
    ranking = sorted(
        table.items(),
        key=lambda x: (
            x[1]["points"],
            x[1]["gd"],
            x[1]["gf"],
            ratings[x[0]]['elo']  # <-- ¡Ojo aquí! Ahora apuntamos a ['elo']
        ),
        reverse=True
    )

    return ranking

# --- PRUEBA DEL GRUPO ---
group = [
    "Spain",
    "Japan",
    "Morocco",
    "Mexico"
]

# Llamamos a la función con el diccionario final y los parámetros óptimos
ranking = simulate_group_hybrid(
    group,
    latest_ratings,
    optimal_params
)

print("Clasificación del Grupo:")
for equipo, stats in ranking:
    print(f"{equipo:<10} | Pts: {stats['points']} | GD: {stats['gd']:>2} | GF: {stats['gf']}")

Clasificación del Grupo:
Morocco    | Pts: 7 | GD:  2 | GF: 5
Spain      | Pts: 7 | GD:  2 | GF: 4
Mexico     | Pts: 3 | GD:  1 | GF: 5
Japan      | Pts: 0 | GD: -5 | GF: 3


# Simulación fase de grupos

In [34]:
groups = {
    "A": ["Mexico", "South Africa", "South Korea", "Czech Republic"],
    "B": ["Canada", "Bosnia and Herzegovina", "Qatar", "Switzerland"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["United States", "Paraguay", "Australia", "Turkey"],

    "E": ["Germany", "Curaçao", "Ivory Coast", "Ecuador"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],

    "I": ["France", "Senegal", "Iraq", "Norway"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "DR Congo", "Uzbekistan", "Colombia"],
    "L": ["England", "Croatia", "Ghana", "Panama"]
}

In [35]:
def simulate_group_stage(groups, ratings, params):
    winners = {}
    runners_up = {}
    thirds = []

    for group_name, teams in groups.items():
        # Llamamos a la nueva función híbrida
        ranking = simulate_group_hybrid(
            teams,
            ratings,
            params
        )

        winner_team, winner_stats = ranking[0]
        runner_team, runner_stats = ranking[1]
        third_team, third_stats = ranking[2]

        winners[group_name] = winner_team
        runners_up[group_name] = runner_team

        thirds.append({
            "group": group_name,
            "team": third_team,
            "points": third_stats["points"],
            "gd": third_stats["gd"],
            "gf": third_stats["gf"]
        })

    return {
        "winners": winners,
        "runners_up": runners_up,
        "thirds": thirds
    }

In [36]:
def select_best_thirds(thirds, ratings):
    ranking = sorted(
        thirds,
        key=lambda x: (
            x["points"],
            x["gd"],
            x["gf"],
            ratings[x["team"]]['elo'] # Desempate extremo por ranking Elo
        ),
        reverse=True
    )

    return ranking[:8]

In [37]:
ranking_H = simulate_group_hybrid(
    groups["H"],
    latest_ratings,
    optimal_params
)

print("--- GRUPO H ---")
for team, stats in ranking_H:
    print(f"{team:<15} {stats}")

--- GRUPO H ---
Spain           {'points': 9, 'gf': np.int64(6), 'ga': 3, 'gd': np.int64(3)}
Uruguay         {'points': 4, 'gf': np.int64(5), 'ga': np.int64(4), 'gd': np.int64(1)}
Saudi Arabia    {'points': 4, 'gf': np.int64(4), 'ga': np.int64(4), 'gd': np.int64(0)}
Cape Verde      {'points': 0, 'gf': 3, 'ga': np.int64(7), 'gd': np.int64(-4)}


In [38]:
group_results = simulate_group_stage(
    groups,
    latest_ratings,
    optimal_params
)

print("\n--- PRIMEROS DE GRUPO ---")
print(group_results["winners"])

print("\n--- SEGUNDOS DE GRUPO ---")
print(group_results["runners_up"])

print("\n--- TERCEROS DE GRUPO (Todos) ---")
for team in group_results["thirds"]:
    print(team)


--- PRIMEROS DE GRUPO ---
{'A': 'Czech Republic', 'B': 'Switzerland', 'C': 'Morocco', 'D': 'Paraguay', 'E': 'Ecuador', 'F': 'Japan', 'G': 'Iran', 'H': 'Spain', 'I': 'France', 'J': 'Austria', 'K': 'Portugal', 'L': 'Croatia'}

--- SEGUNDOS DE GRUPO ---
{'A': 'Mexico', 'B': 'Canada', 'C': 'Brazil', 'D': 'Turkey', 'E': 'Ivory Coast', 'F': 'Tunisia', 'G': 'Belgium', 'H': 'Saudi Arabia', 'I': 'Norway', 'J': 'Algeria', 'K': 'Uzbekistan', 'L': 'England'}

--- TERCEROS DE GRUPO (Todos) ---
{'group': 'A', 'team': 'South Korea', 'points': 4, 'gd': np.int64(0), 'gf': np.int64(2)}
{'group': 'B', 'team': 'Qatar', 'points': 3, 'gd': np.int64(-2), 'gf': np.int64(3)}
{'group': 'C', 'team': 'Scotland', 'points': 3, 'gd': np.int64(-2), 'gf': np.int64(3)}
{'group': 'D', 'team': 'Australia', 'points': 2, 'gd': np.int64(-1), 'gf': np.int64(0)}
{'group': 'E', 'team': 'Germany', 'points': 4, 'gd': np.int64(0), 'gf': np.int64(2)}
{'group': 'F', 'team': 'Sweden', 'points': 3, 'gd': np.int64(0), 'gf': np.int64(

In [39]:
best_thirds = select_best_thirds(
    group_results["thirds"], 
    latest_ratings
)

print("\n--- LOS 8 MEJORES TERCEROS CLASIFICADOS ---")
for team in best_thirds:
    print(team)

qualified_groups = sorted([
    x["group"]
    for x in best_thirds
])

print(f"\nGrupos de los terceros clasificados: {qualified_groups}")


--- LOS 8 MEJORES TERCEROS CLASIFICADOS ---
{'group': 'K', 'team': 'Colombia', 'points': 4, 'gd': np.int64(1), 'gf': np.int64(5)}
{'group': 'H', 'team': 'Uruguay', 'points': 4, 'gd': np.int64(1), 'gf': np.int64(4)}
{'group': 'E', 'team': 'Germany', 'points': 4, 'gd': np.int64(0), 'gf': np.int64(2)}
{'group': 'A', 'team': 'South Korea', 'points': 4, 'gd': np.int64(0), 'gf': np.int64(2)}
{'group': 'G', 'team': 'Egypt', 'points': 4, 'gd': np.int64(-1), 'gf': np.int64(6)}
{'group': 'I', 'team': 'Iraq', 'points': 3, 'gd': np.int64(1), 'gf': np.int64(3)}
{'group': 'J', 'team': 'Argentina', 'points': 3, 'gd': np.int64(0), 'gf': np.int64(4)}
{'group': 'F', 'team': 'Sweden', 'points': 3, 'gd': np.int64(0), 'gf': np.int64(3)}

Grupos de los terceros clasificados: ['A', 'E', 'F', 'G', 'H', 'I', 'J', 'K']


# Seleccionar mejores terceros

In [40]:
def get_qualified_groups(best_thirds):

    return tuple(sorted(
        x["group"]
        for x in best_thirds
    ))

In [41]:
third_pattern = get_qualified_groups(
    best_thirds
)

print(third_pattern)

('A', 'E', 'F', 'G', 'H', 'I', 'J', 'K')


In [42]:
third_mapping_df = pd.read_csv(
    "../data/combinaciones_terceros_mundial_2026.csv"
)

In [43]:
def get_third_place_mapping(
    best_thirds,
    mapping_df
):

    qualified = sorted(
        x["group"]
        for x in best_thirds
    )

    qualified_str = ",".join(
        qualified
    )

    row = mapping_df[
        mapping_df[
            "Grupos_Clasificados"
        ] == qualified_str
    ]

    if len(row) == 0:

        raise ValueError(
            f"No mapping found for {qualified_str}"
        )

    return row.iloc[0]

In [44]:
mapping = get_third_place_mapping(
    best_thirds,
    third_mapping_df
)

print(mapping)

Combinacion_Id                     323
Grupos_Clasificados    A,E,F,G,H,I,J,K
Grupos_Eliminados              B,C,D,L
Rival_de_1A                         3E
Rival_de_1B                         3F
Rival_de_1D                         3I
Rival_de_1E                         3A
Rival_de_1G                         3H
Rival_de_1I                         3G
Rival_de_1K                         3J
Rival_de_1L                         3K
Name: 322, dtype: object


# Crear cuadro Knockout

In [45]:
def build_third_lookup(
    best_thirds
):

    return {
        x["group"]: x["team"]
        for x in best_thirds
    }

In [46]:
def build_round_of_32(
    group_results,
    best_thirds,
    mapping_row
):

    winners = group_results["winners"]
    runners = group_results["runners_up"]

    third_lookup = {
        x["group"]: x["team"]
        for x in best_thirds
    }

    matches = {}

    # Partido 73
    matches[73] = (
        runners["A"],
        runners["B"]
    )

    # Partido 74
    matches[74] = (
        winners["E"],
        third_lookup[
            mapping_row["Rival_de_1E"][1]
        ]
    )

    # Partido 75
    matches[75] = (
        winners["F"],
        runners["C"]
    )

    # Partido 76
    matches[76] = (
        winners["C"],
        runners["F"]
    )

    # Partido 77
    matches[77] = (
        winners["I"],
        third_lookup[
            mapping_row["Rival_de_1I"][1]
        ]
    )

    # Partido 78
    matches[78] = (
        runners["E"],
        runners["I"]
    )

    # Partido 79
    matches[79] = (
        winners["A"],
        third_lookup[
            mapping_row["Rival_de_1A"][1]
        ]
    )

    # Partido 80
    matches[80] = (
        winners["L"],
        third_lookup[
            mapping_row["Rival_de_1L"][1]
        ]
    )

    # Partido 81
    matches[81] = (
        winners["D"],
        third_lookup[
            mapping_row["Rival_de_1D"][1]
        ]
    )

    # Partido 82
    matches[82] = (
        winners["G"],
        third_lookup[
            mapping_row["Rival_de_1G"][1]
        ]
    )

    # Partido 83
    matches[83] = (
        runners["K"],
        runners["L"]
    )

    # Partido 84
    matches[84] = (
        winners["H"],
        runners["J"]
    )

    # Partido 85
    matches[85] = (
        winners["B"],
        third_lookup[
            mapping_row["Rival_de_1B"][1]
        ]
    )

    # Partido 86
    matches[86] = (
        winners["J"],
        runners["H"]
    )

    # Partido 87
    matches[87] = (
        winners["K"],
        third_lookup[
            mapping_row["Rival_de_1K"][1]
        ]
    )

    # Partido 88
    matches[88] = (
        runners["D"],
        runners["G"]
    )

    return matches

In [47]:

round32 = build_round_of_32(
    group_results,
    best_thirds,
    mapping
)

print(len(round32))

16


# Simulación Knockout

In [48]:
def simulate_knockout_match(team_a, team_b, ratings, params):
    # 1. Llamamos a nuestra nueva función híbrida
    probs = predict_match_final(
        team_home=team_a,
        team_away=team_b,
        ratings_dict=ratings,
        params=params,
        neutral=True
    )

    # 2. En eliminatorias, si empatan, se asume un 50/50 en la prórroga/penaltis
    pass_a = probs["home_win"] + 0.5 * probs["draw"]
    pass_b = probs["away_win"] + 0.5 * probs["draw"]

    # 3. Normalizamos para evitar errores de coma flotante de Numpy
    total = pass_a + pass_b

    # 4. Elegimos ganador
    winner = np.random.choice(
        [team_a, team_b],
        p=[pass_a / total, pass_b / total]
    )

    return winner

In [49]:
def play_match_dict(matches, ratings, params):
    winners = {}

    for match_id, match in matches.items():
        team_a, team_b = match
        # Usamos la función actualizada
        winners[match_id] = simulate_knockout_match(
            team_a,
            team_b,
            ratings,
            params
        )

    return winners

In [50]:
def simulate_knockout_round(
    matches,
    ratings,
    model
):

    winners = []

    for team_a, team_b in matches:

        winner = simulate_knockout_match(
            model,
            ratings,
            team_a,
            team_b
        )

        winners.append(
            winner
        )

    return winners

In [51]:
def build_round_of_16(
    winners32
):

    return {

        89: (
            winners32[74],
            winners32[77]
        ),

        90: (
            winners32[73],
            winners32[75]
        ),

        91: (
            winners32[76],
            winners32[78]
        ),

        92: (
            winners32[79],
            winners32[80]
        ),

        93: (
            winners32[83],
            winners32[84]
        ),

        94: (
            winners32[81],
            winners32[82]
        ),

        95: (
            winners32[86],
            winners32[88]
        ),

        96: (
            winners32[85],
            winners32[87]
        )
    }

In [52]:
def build_quarters(
    winners16
):

    return {

        97: (
            winners16[89],
            winners16[90]
        ),

        98: (
            winners16[93],
            winners16[94]
        ),

        99: (
            winners16[91],
            winners16[92]
        ),

        100: (
            winners16[95],
            winners16[96]
        )
    }

In [53]:
def build_semis(
    winners8
):

    return {

        101: (
            winners8[97],
            winners8[98]
        ),

        102: (
            winners8[99],
            winners8[100]
        )
    }

In [54]:
def build_final(
    winners4
):

    return {

        104: (
            winners4[101],
            winners4[102]
        )
    }

In [55]:
def simulate_world_cup(groups, ratings, params, mapping_df):
    
    # --- FASE DE GRUPOS ---
    group_results = simulate_group_stage(
        groups,
        ratings,
        params
    )

    # --- MEJORES TERCEROS ---
    # ¡Importante! Pasamos 'ratings' para el desempate por Elo
    best_thirds = select_best_thirds(
        group_results["thirds"],
        ratings
    )

    # --- BUSCAR FILA FIFA ---
    mapping = get_third_place_mapping(
        best_thirds,
        mapping_df
    )

    # --- DIECISEISAVOS (73-88) ---
    round32 = build_round_of_32(
        group_results,
        best_thirds,
        mapping
    )
    winners32 = play_match_dict(round32, ratings, params)

    # --- OCTAVOS (89-96) ---
    round16 = build_round_of_16(winners32)
    winners16 = play_match_dict(round16, ratings, params)

    # --- CUARTOS (97-100) ---
    quarters = build_quarters(winners16)
    winners8 = play_match_dict(quarters, ratings, params)

    # --- SEMIS (101-102) ---
    semis = build_semis(winners8)
    winners4 = play_match_dict(semis, ratings, params)

    # --- FINAL (104) ---
    final = build_final(winners4)
    champion = play_match_dict(final, ratings, params)[104]

    return champion

In [56]:
# Hacemos una prueba rápida de una simulación
campeon = simulate_world_cup(
    groups,
    latest_ratings,   # Tu foto final de ratings (con Elo y ODM)
    optimal_params,   # Tus betas y k0 optimizados
    third_mapping_df
)

print(f"El ganador de la simulación de prueba es: {campeon}")

El ganador de la simulación de prueba es: France


# Simulación final Montecarlo

In [57]:
from collections import Counter
import pandas as pd

counter = Counter()

N = 10000

print(f"Iniciando {N} simulaciones del Mundial 2026...")

for i in range(N):
    
    # Llamamos a la función con los NUEVOS argumentos
    champion = simulate_world_cup(
        groups,
        latest_ratings,     # El nuevo diccionario con Elo, Ataque y Defensa
        optimal_params,     # Tus parámetros descubiertos por L-BFGS-B
        third_mapping_df
    )

    counter[champion] += 1

    # Imprimir progreso para no desesperar
    if (i + 1) % 1000 == 0:
        print(f"Progreso: {i + 1} simulaciones completadas...")

# Preparar los resultados
results = []
for team, wins in counter.items():
    results.append({
        "team": team,
        "probability": wins / N
    })

# Ordenar de mayor a menor probabilidad
results = sorted(
    results,
    key=lambda x: x["probability"],
    reverse=True
)

# Imprimir el TOP 20
print("\n=== PROBABILIDADES DE GANAR EL MUNDIAL 2026 ===")
for row in results[:20]:
    print(f"{row['team']:<20s} {100 * row['probability']:>5.2f}%")

Iniciando 10000 simulaciones del Mundial 2026...
Progreso: 1000 simulaciones completadas...
Progreso: 2000 simulaciones completadas...
Progreso: 3000 simulaciones completadas...
Progreso: 4000 simulaciones completadas...
Progreso: 5000 simulaciones completadas...
Progreso: 6000 simulaciones completadas...
Progreso: 7000 simulaciones completadas...
Progreso: 8000 simulaciones completadas...
Progreso: 9000 simulaciones completadas...
Progreso: 10000 simulaciones completadas...

=== PROBABILIDADES DE GANAR EL MUNDIAL 2026 ===
Spain                20.40%
England              14.40%
France               14.14%
Argentina            11.28%
Colombia              6.95%
Portugal              5.55%
Morocco               4.29%
Norway                3.54%
Ecuador               2.65%
Croatia               2.57%
Japan                 1.94%
Netherlands           1.90%
Switzerland           1.60%
Austria               1.48%
Canada                1.13%
Brazil                1.13%
Belgium               0

In [58]:
print(
    pd.DataFrame(results)
    .sort_values(
        "probability",
        ascending=False
    )
    .head(30)
)

             team  probability
0           Spain       0.2040
1         England       0.1440
2          France       0.1414
3       Argentina       0.1128
4        Colombia       0.0695
5        Portugal       0.0555
6         Morocco       0.0429
7          Norway       0.0354
8         Ecuador       0.0265
9         Croatia       0.0257
10          Japan       0.0194
11    Netherlands       0.0190
12    Switzerland       0.0160
13        Austria       0.0148
14         Canada       0.0113
15         Brazil       0.0113
16        Belgium       0.0095
17        Germany       0.0082
18         Turkey       0.0077
19      Australia       0.0058
20        Algeria       0.0032
21         Mexico       0.0024
22           Iran       0.0021
23  United States       0.0019
24       Paraguay       0.0017
25     Uzbekistan       0.0014
26        Senegal       0.0013
27        Uruguay       0.0012
28    Ivory Coast       0.0011
29         Jordan       0.0009


# Antiguo Modelo

In [16]:
import pandas as pd


def encode_result(
    home_score,
    away_score
):

    if home_score > away_score:
        return 2

    if home_score < away_score:
        return 0

    return 1


def create_training_dataset(
    historical_df,
    elo_history
):

    df = historical_df.merge(

        elo_history,

        on=[
            "date",
            "home_team",
            "away_team"
        ],

        how="inner"
    )

    df["elo_diff"] = (df["home_elo"]- df["away_elo"]
)
    df["elo_diff_scaled"] = (df["home_elo"]- df["away_elo"]) / 100
    
    df["result"] = df.apply(

        lambda row: encode_result(
            row["home_score"],
            row["away_score"]
        ),

        axis=1
    )

    return df

In [17]:
training_df = (
    create_training_dataset(
        historical_df,
        elo_history
    )
)

In [18]:
training_df

,date,home_team,away_team,home_score,away_score,tournament_x,city,country,neutral_x,home_elo,away_elo,tournament_y,neutral_y,elo_diff,elo_diff_scaled,result
0,2018-01-02,Oman,Bahrain,1,0,Gulf Cup,Kuwait City,Kuwait,True,1500.000000,1500.000000,Gulf Cup,True,0.000000,0.000000,2
1,2018-01-02,Iraq,United Arab Emirates,0,0,Gulf Cup,Kuwait City,Kuwait,True,1500.000000,1500.000000,Gulf Cup,True,0.000000,0.000000,1
2,2018-01-05,Oman,United Arab Emirates,0,0,Gulf Cup,Kuwait City,Kuwait,True,1507.298265,1500.000000,Gulf Cup,True,7.298265,0.072983,1
3,2018-01-07,Estonia,Sweden,1,1,Friendly,Abu Dhabi,United Arab Emirates,True,1500.000000,1500.000000,Friendly,True,0.000000,0.000000,1
4,2018-01-11,Denmark,Sweden,0,1,Friendly,Abu Dhabi,United Arab Emirates,True,1500.000000,1500.000000,Friendly,True,0.000000,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7980,2026-05-31,Germany,Finland,4,0,Friendly,Mainz,Germany,False,1740.516695,1452.251311,Friendly,False,288.265385,2.882654,2
7981,2026-05-31,Poland,Ukraine,0,2,Friendly,Wrocław,Poland,False,1563.473272,1605.982948,Friendly,False,-42.509676,-0.425097,0
7982,2026-05-31,Singapore,Mongolia,4,0,Friendly,Singapore,Singapore,False,1444.639293,1337.746075,Friendly,False,106.893218,1.068932,2
7983,2026-05-31,Switzerland,Jordan,4,1,Friendly,St. Gallen,Switzerland,False,1698.080551,1688.987732,Friendly,False,9.092819,0.090928,2


In [19]:
print(
    training_df[
        [
            "home_team",
            "away_team",
            "home_elo",
            "away_elo",
            "elo_diff",
            "elo_diff_scaled",
            "neutral_x",
            "result"
        ]
    ].head()
)

  home_team             away_team     home_elo  away_elo  elo_diff  \
0      Oman               Bahrain  1500.000000    1500.0  0.000000   
1      Iraq  United Arab Emirates  1500.000000    1500.0  0.000000   
2      Oman  United Arab Emirates  1507.298265    1500.0  7.298265   
3   Estonia                Sweden  1500.000000    1500.0  0.000000   
4   Denmark                Sweden  1500.000000    1500.0  0.000000   

   elo_diff_scaled  neutral_x  result  
0         0.000000       True       2  
1         0.000000       True       1  
2         0.072983       True       1  
3         0.000000       True       1  
4         0.000000       True       0  


In [20]:
print(
    training_df["result"]
    .value_counts(normalize=True)
)

result
2    0.477145
0    0.292423
1    0.230432
Name: proportion, dtype: float64


## Entrenamiento de modelo

In [21]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss


def train_model(
    train_df,
    test_df
):

    X_train = train_df[
        [
            "elo_diff_scaled",
            "neutral_x"
        ]
    ]

    y_train = train_df["result"]

    X_test = test_df[
        [
            "elo_diff_scaled",
            "neutral_x"
        ]
    ]

    y_test = test_df["result"]

    model = LogisticRegression(
        
        max_iter=1000,
        random_state=42
    )

    model.fit(
        X_train,
        y_train
    )

    probs = model.predict_proba(
        X_test
    )

    score = log_loss(
        y_test,
        probs
    )

    print(
        f"Log Loss: {score:.4f}"
    )

    return model

In [22]:
cutoff_date = "2022-01-01"

train_df = training_df[
    training_df["date"] < cutoff_date
]

test_df = training_df[
    training_df["date"] >= cutoff_date
]

In [23]:
print(train_df.shape)
print(test_df.shape)

(3540, 16)
(4445, 16)


In [24]:
model = train_model(
    train_df,
    test_df
)

Log Loss: 0.9521


In [25]:
print(model.coef_)
print(model.intercept_)

[[-0.84280892  0.30714303]
 [ 0.0076021  -0.01890846]
 [ 0.83520681 -0.28823457]]
[-0.23338102 -0.217974    0.45135502]


## Predictor

In [26]:
import pandas as pd
MATCH_CACHE = {}

def predict_match(
    model,
    home_elo,
    away_elo,
    neutral=True
):

    key = (
        round(home_elo, 1),
        round(away_elo, 1),
        neutral
    )

    if key in MATCH_CACHE:
        return MATCH_CACHE[key]
    
    elo_diff = (
    home_elo - away_elo
    )

    X = pd.DataFrame({
        "elo_diff_scaled": [
            elo_diff / 100
        ],
        "neutral_x": [
            neutral
        ]
    })

    probs = model.predict_proba(X)[0]

    result = {
    "away_win": probs[0],
    "draw": probs[1],
    "home_win": probs[2]
    }

    MATCH_CACHE[key] = result

    return result

In [29]:
print(model.coef_)
print(model.intercept_)

[[-0.84280892  0.30714303]
 [ 0.0076021  -0.01890846]
 [ 0.83520681 -0.28823457]]
[-0.23338102 -0.217974    0.45135502]
